# National Coverage Extension — DPMA register → NUTS regions

**The gap.** A NUTS region filter in PATSTAT returns only the **EP/PCT-active subset** of a
region's applicants — NUTS is assigned on the EP/PCT route only. ~70 % of German national
patent families never take that route and carry **no NUTS**; PATSTAT can't recover them by
postcode either (`tls226.zip_code` is empty). Those national-only filers — typically the
smaller, locally-filing SMEs a PATLIB most wants to reach — are invisible.

**The fix shown here.** The DPMA register *does* carry every national filing's applicant
**address incl. PLZ**. This notebook fetches/parses that register data and maps the PLZ to the
**same NUTS3 code PATSTAT uses**, so the national-only tail becomes comparable to the
EP/PCT population. Three small helpers in `dpma/`:

| helper | does |
|---|---|
| `dpma.fetch` | authenticated REST client (`search` + `getRegisterInfo`) |
| `dpma.register_parser` | ST.36 XML → applicant rows (name, PLZ, city, kind, IPC, …) |
| `dpma.plz_nuts` | PLZ → NUTS3 + Bundesland (Eurostat crosswalk) |

Design & caveats: [`docs/national-coverage-extension-dpmaconnect.md`](docs/national-coverage-extension-dpmaconnect.md).

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Make the `dpma` helper package importable regardless of where Jupyter launched.
ROOT = next(c for c in [Path.cwd(), *Path.cwd().parents]
            if (c / "dpma" / "register_parser.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dpma import parse_register_xml, applicant_rows, enrich_rows, fetch, plz_nuts

SAMPLES = ROOT / "dpma" / "samples"
print("helpers loaded from", ROOT / "dpma")

## Part 1 — Offline demo (bundled sample records)

Runs without credentials or network. Two real register records ship under `dpma/samples/`
(a 1977 utility model and a 2024 patent; inventor personal data was stripped for GDPR).
We parse them into applicant rows and attach the NUTS region from the PLZ.

In [ ]:
rows = []
for xml in sorted(SAMPLES.glob("*.xml")):
    rows += applicant_rows(parse_register_xml(xml))

enrich_rows(rows)   # adds nuts3 / nuts1 / bundesland from the PLZ

cols = ["name", "plz", "city", "country", "nuts3", "bundesland",
        "kind", "appln_number", "filing_date", "ipc"]
pd.DataFrame(rows, columns=cols)

The `nuts3` and `bundesland` columns are exactly what PATSTAT would leave empty for these
national-only filings — recovered here from the register PLZ. `kind` is `A` (patent) or
`U` (utility model).

## Part 2 — Live fetch (optional)

Needs `DPMA_USER` / `DPMA_PASS` in the environment **and** network egress to
`dpmaconnect.dpma.de`. The applicant field is `INH` (Inhaber). `search` is capped at **1000
hits** — for a whole Bundesland/year use the bulk routes (`getRegisterabzuege` /
`getPublikationsdaten_XML`), not `search`. The cell degrades gracefully if unavailable.

In [ ]:
APPLICANT = "Hager"     # try any company name
MAX_RECORDS = 15         # keep the demo small; raise for real work

live_rows = []
try:
    client = fetch.DpmaClient()                       # reads DPMA_USER / DPMA_PASS
    hits = client.search_applicant(APPLICANT)
    print(f"{len(hits)} hits for INH={APPLICANT!r} "
          f"(showing first {MAX_RECORDS}; search caps at 1000)")
    for h in hits[:MAX_RECORDS]:
        reg = parse_register_xml(client.get_register_info(h.leading_registered_number))
        live_rows += applicant_rows(reg)
    enrich_rows(live_rows)
    display(pd.DataFrame(live_rows)[["name", "plz", "city", "bundesland",
                                     "nuts3", "kind", "appln_number"]])
except Exception as e:
    print("Live fetch skipped:", type(e).__name__, "-", e)

## Part 3 — Aggregate by region

Whatever rows we have (live if fetched, else the samples), count applicant records per
region — the regional lead list PATSTAT could not build for national-only filers.

In [ ]:
work = pd.DataFrame(live_rows) if live_rows else pd.DataFrame(rows)
de = work[work["country"] == "DE"]

by_region = (de.groupby(["bundesland", "nuts3"], dropna=False)
               .size().reset_index(name="applicant_records")
               .sort_values("applicant_records", ascending=False))
by_region

## Caveats

- **Scope:** this recovers the *national-only tail*; the full picture is the **union** of the
  PATSTAT NUTS list (EP/PCT-active) **and** these DPMA national-only firms. Joining a firm to
  its PATSTAT families (Aktenzeichen ↔ `appln_nr`, `has_EP` flag) is the next step — number
  normalisation is still open (see the design doc).
- **GDPR:** company addresses are low-risk B2B; **natural-person** applicant addresses are
  personal data — handle accordingly.
- **Foreign applicants** carry no German PLZ, so `nuts3` is `None` for them (by design).
- **DE only** — the analogous national route for FR is INPI.